# МУС Training — Google Colab
Автоматическая компиляция, тренировка и выгрузка на Hugging Face

In [ ]:
# @title 1. Setup
import os, sys, subprocess, json, time
from pathlib import Path

HF_TOKEN = ""  # @param {type:"string"}
MODEL = "400m"  # @param ["400m", "700m", "1b"]
EPOCHS = 5  # @param {type:"integer"}

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["MUS_MODEL"] = MODEL
os.environ["MUS_EPOCHS"] = str(EPOCHS)

print(f"Model: {MODEL}, Epochs: {EPOCHS}")

In [ ]:
# @title 2. Clone & Build
!echo "Cloning MUS source..."
!rm -rf mus-source
!git clone https://github.com/Shuteira/mus-cuda.git mus-source 2>&1 | tail -5

%cd /content/mus-source
!mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release -DMUS_BUILD_CLOUD=ON -DMUS_BUILD_MINIMAL=ON 2>&1 | tail -10
!cd build && make -j$(nproc) mus_train_kaggle 2>&1 | tail -10
!ls -lh build/mus_train_kaggle

In [ ]:
# @title 3. Train
!echo "Training {MODEL} for {EPOCHS} epochs..."
!./build/mus_train_kaggle {MODEL} --epochs {EPOCHS} 2>&1 | tail -50

In [ ]:
# @title 4. Upload to Hugging Face
!pip install huggingface_hub -q

from huggingface_hub import HfApi
import glob

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    weights = list(Path("/content/mus-source/weights").glob("*.bin"))
    if weights:
        w = weights[-1]
        name = f"mus-{MODEL}-colab-{int(time.time())}.bin"
        api.upload_file(path_or_fileobj=str(w), path_in_repo=f"weights/{name}",
                       repo_id="Shuteira/mus-uran-weights", repo_type="model")
        print(f"Uploaded: {name}")
    else:
        print("No weights found!")
else:
    print("No HF token, skipping upload")